# 08 The decision layer

**Purpose:** turn the model outputs from notebooks 05 to 07 into what each person at Waypoint actually needs to act on. The dispatcher, loaders, drivers and store managers are not data people. A late probability of 0.63 or a forecast error of 7% means nothing to a driver at 3 AM.

**Rules every output follows:**
- **Clock times, not probabilities.** "Arrives 06:10 to 06:40", not "P(late) = 0.31".
- **Ranges, not single numbers**, so nobody is misled by false precision.
- **A traffic light always comes with an action.**
- **One reason line**, taken from the features that pushed the risk up the most.

**Who gets what:**

| Role | When | Output | Section |
|---|---|---|---|
| Dispatcher | after the 4 PM cutoff | risk board: every route with red, amber and green stops, a reason and a suggested fix | 2 |
| Driver | before leaving the depot | run sheet: windows, arrival ranges, unloading minutes, risky stops | 3 |
| Store manager | the evening before | a short message with the arrival range | 3, 4 |
| Dispatcher | on a peak day | deferral list with reasons, and a notice for each store | 4 |
| Loader | before loading | load order per trip, chilled section, how full the vehicle is | 4 |
| Fleet manager | weekly | reefer outlook for the next 10 weeks, and when to book reefer maintenance | 5 |

**Inputs:** the saved Task 1 models, the Task 1 and 2A predictions, and the Task 2B plan from notebook 07.

**Outputs:** sample sheets and messages in `reports/decision_layer/`, ready to show the people who would use them.

In [1]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xora import allocation as al
from xora import decisions as dl
from xora import features as fe
from xora.io import load_processed, load_raw
from xora.paths import MODELS, REPORTS, SUBMISSIONS
from xora.plotting import apply_style, SERIES, INK, INK_SOFT

apply_style()
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 110)

OUT = REPORTS / "decision_layer"
OUT.mkdir(parents=True, exist_ok=True)
LIGHTS = {"red": "#f6d5cf", "amber": "#fbe8c0", "green": "#d6efe2"}

def paint(df, col="light"):
    return df.style.apply(lambda r: [f"background-color: {LIGHTS.get(r[col], '')}"] * len(r), axis=1).hide(axis="index")

## 1. Rebuild the Task 1 view of the test period

The saved models are reloaded exactly as the final notebook will reload them. The route simulator gives each stop an arrival range (the middle and the slow end of 1,000 replays), the submission gives the late risk, and the challenger classifier splits each stop's risk into one contribution per feature, which becomes the reason line.

**Thresholds.** Notebook 05 showed the late-risk blend is calibrated on the holdout: across risk bands, the share of stops that ran late matched the predicted risk. So the lights can be set in plain terms:
- **red** at 50% or more: at least half of stops like this run late
- **amber** from 25%: about one in three or four
- **green** below 25%

In [2]:
stops = load_processed("stops")
tables = dict(outlets=load_processed("outlets"), vehicles=load_processed("vehicles"),
              allowance=load_processed("service_allowance"), calendar=load_processed("calendar"),
              road=load_processed("road_conditions"), traffic=load_processed("traffic_speed"))
full = fe.model_frame(stops, fe.build_task1_features(stops, **tables))
test = full[full.split == "test"]

simulator = joblib.load(MODELS / "route_simulator.pkl")
classifier = joblib.load(MODELS / "late_classifier.pkl")
task1 = pd.read_csv(SUBMISSIONS / "submission_task1.csv")

bands = simulator.simulate(test)[["delivery_id", "arrival_p50_min", "arrival_p90_min"]]
test = test.merge(task1, on="delivery_id").merge(bands, on="delivery_id")
feats = classifier.feature_name_
contrib = pd.DataFrame(classifier.booster_.predict(test[feats], pred_contrib=True)[:, :-1], columns=feats, index=test.index)

test["light"] = test.pred_late_prob.map(dl.light)
test["reason"] = [dl.reason_line(test.loc[i], contrib.loc[i]) if test.at[i, "light"] != "green" else ""
                  for i in test.index]
test["arrives"] = [dl.arrival_range(a, b) for a, b in zip(test.arrival_p50_min, test.arrival_p90_min)]

share = test.light.value_counts().reindex(["green", "amber", "red"])
pd.DataFrame({"stops": share, "share": (share / len(test) * 100).round(1).astype(str) + "%",
              "expected late": test.groupby("light").pred_late_prob.sum().reindex(share.index).round(0)})

,stops,share,expected late
light,,,
green,3658,73.0%,96.0
amber,307,6.1%,108.0
red,1049,20.9%,899.0


Across the six test weeks about one stop in five is red, and the red stops carry most of the expected late arrivals. That is the point of the board: the dispatcher looks at a fifth of the stops and finds most of the trouble.

## 2. Dispatcher: the risk board

The board is built at the 4 PM cutoff for the next day's draft routes. We show the riskiest test day, 7 March 2026, a monsoon Saturday.

One row per route, worst first, with a suggested fix that depends on where the red stops sit:
- red from the first stop: the whole route starts behind, so **leave earlier**
- red only further down: **split the route** before the first red stop, or move those stops to the front

In [3]:
DAY = test.groupby("date").pred_late_prob.mean().idxmax()
day = test[test.date == DAY].sort_values(["route_id", "seq"])

board = []
for rid, g in day.groupby("route_id"):
    worst = g.loc[g.pred_late_prob.idxmax()]
    board.append({"route": rid, "depot": g.depot.iloc[0], "brand": g.brand.iloc[0], "district": g.district.iloc[0],
                  "stops": len(g), "red": int((g.light == "red").sum()), "amber": int((g.light == "amber").sum()),
                  "leaves": dl.clock(g.route_start_min.iloc[0]),
                  "worst stop": f"stop {int(worst.seq) + 1} ({worst.outlet_id})" if worst.light != "green" else "",
                  "why": worst.reason, "suggested fix": dl.route_fix(g)})
board = pd.DataFrame(board)
board["light"] = np.where(board.red > 0, "red", np.where(board.amber > 0, "amber", "green"))
board = board.sort_values(["red", "amber"], ascending=False)
board.to_csv(OUT / "dispatcher_risk_board.csv", index=False)
print(f"{DAY.date()}: {len(board)} routes, {(board.light == 'red').sum()} with at least one red stop")
paint(board.head(12))

2026-03-07: 36 routes, 25 with at least one red stop


route,depot,brand,district,stops,red,amber,leaves,worst stop,why,suggested fix,light
R025904,Peliyagoda,Fresh,Colombo,8,5,0,05:01,stop 7 (OUT011),about 87 min past the cut-off once usual traffic is counted; the plan leaves 13 min before the cut-off,"split the route after stop 3, or move the red stops to the front",red
R025878,Peliyagoda,Fresh,Kurunegala,4,4,0,02:00,stop 4 (OUT069),"road advisory for Kurunegala (46, 100 = clear); about 16 min past the cut-off once usual traffic is counted",leave earlier: the route is at risk from its first stop,red
R025895,Peliyagoda,Fresh,Kurunegala,4,4,0,03:34,stop 3 (OUT068),"road advisory for Kurunegala (46, 100 = clear); about 71 min past the cut-off once usual traffic is counted",leave earlier: the route is at risk from its first stop,red
R025874,Kandy,Fresh,Kandy,6,3,1,05:22,stop 6 (OUT081),"about 46 min past the cut-off once usual traffic is counted; road advisory for Kandy (59, 100 = clear)","split the route after stop 3, or move the red stops to the front",red
R025886,Peliyagoda,Fresh,Colombo,7,3,0,04:39,stop 7 (OUT014),about 57 min past the cut-off once usual traffic is counted; OUT014 is often slow to receive,"split the route after stop 4, or move the red stops to the front",red
R025888,Peliyagoda,Fresh,Colombo,3,3,0,06:40,stop 2 (OUT003),about 51 min past the cut-off once usual traffic is counted; planned to arrive after the cut-off,leave earlier: the route is at risk from its first stop,red
R025900,Kandy,Fresh,Kegalle,4,3,0,05:32,stop 4 (OUT119),about 101 min past the cut-off once usual traffic is counted; the plan leaves 5 min before the cut-off,"split the route after stop 1, or move the red stops to the front",red
R025901,Peliyagoda,Fresh,Gampaha,5,3,0,05:02,stop 5 (OUT031),"about 17 min past the cut-off once usual traffic is counted; road advisory for Gampaha (61, 100 = clear)","split the route after stop 2, or move the red stops to the front",red
R025897,Kandy,Fresh,Nuwara Eliya,5,2,1,03:23,stop 5 (OUT107),about 195 min past the cut-off once usual traffic is counted; OUT107 is often slow to receive,"split the route after stop 3, or move the red stops to the front",red
R025899,Peliyagoda,Fresh,Kalutara,7,2,1,03:43,stop 7 (OUT045),about 86 min past the cut-off once usual traffic is counted; OUT045 is often slow to receive,"split the route after stop 5, or move the red stops to the front",red


Each line is something the dispatcher can do before the trucks leave. Reading the reasons down the board, the same few causes come up again and again: a route that is already behind once the usual traffic is counted, and road advisories on the day. Those are exactly the drivers notebook 05 found.

## 3. Driver run sheet and store messages

The driver of the riskiest route gets a run sheet on their phone before leaving the depot, so it works without signal. Each store on the route gets a message the evening before.

In [4]:
ROUTE = board.route.iloc[0]
r = day[day.route_id == ROUTE].sort_values("seq")
run_sheet = pd.DataFrame({
    "stop": r.seq.astype(int) + 1, "outlet": r.outlet_id,
    "window": [f"{dl.clock(a)} to {dl.clock(b)}" for a, b in zip(r.window_open_time_min, r.window_close_time_min)],
    "planned": r.planned_arrival_min.map(dl.clock), "likely arrival": r.arrives,
    "unloading": [f"about {5 * max(1, round(m / 5))} min" for m in r.pred_service_min],
    "light": r.light,
    "note": np.where(r.light == "red", "likely late: call the store before you set off for it",
                     np.where(r.light == "amber", "call ahead if you are running behind", "")),
})
run_sheet.to_csv(OUT / f"driver_run_sheet_{ROUTE}.csv", index=False)
print(f"Route {ROUTE}, {r.brand.iloc[0]} {r.district.iloc[0]}, leaves the depot {dl.clock(r.route_start_min.iloc[0])}")
paint(run_sheet)

Route R025904, Fresh Colombo, leaves the depot 05:01


stop,outlet,window,planned,likely arrival,unloading,light,note
1,OUT008,05:00 to 07:30,05:26,06:00 to 06:15,about 10 min,green,
2,OUT010,05:00 to 07:30,05:48,06:25 to 06:45,about 20 min,green,
3,OUT013,05:00 to 07:30,06:12,07:15 to 07:40,about 15 min,green,
4,OUT005,04:00 to 07:45,06:36,08:00 to 08:30,about 20 min,red,likely late: call the store before you set off for it
5,OUT006,03:00 to 08:00,06:59,08:50 to 09:25,about 25 min,red,likely late: call the store before you set off for it
6,OUT007,05:30 to 08:00,07:23,09:40 to 10:20,about 30 min,red,likely late: call the store before you set off for it
7,OUT011,03:00 to 08:00,07:47,10:35 to 11:20,about 25 min,red,likely late: call the store before you set off for it
8,OUT012,05:30 to 08:00,08:10,11:25 to 12:15,about 20 min,red,likely late: call the store before you set off for it


In [5]:
messages = pd.DataFrame({"outlet": r.outlet_id, "light": r.light, "message": [dl.store_message(x) for x in r.itertuples()]})
messages.to_csv(OUT / f"store_messages_{ROUTE}.csv", index=False)
paint(messages)

outlet,light,message
OUT008,green,"Your Fresh delivery arrives 06:00 to 06:15, usually on time."
OUT010,green,"Your Fresh delivery arrives 06:25 to 06:45, usually on time."
OUT013,green,"Your Fresh delivery arrives 07:15 to 07:40, usually on time."
OUT005,red,"Your Fresh delivery is planned for 08:00 to 08:30. It may arrive after your 07:45 cut-off, so the driver will call you on the way. Please keep the receiving bay free."
OUT006,red,"Your Fresh delivery is planned for 08:50 to 09:25. It may arrive after your 08:00 cut-off, so the driver will call you on the way. Please keep the receiving bay free."
OUT007,red,"Your Fresh delivery is planned for 09:40 to 10:20. It may arrive after your 08:00 cut-off, so the driver will call you on the way. Please keep the receiving bay free."
OUT011,red,"Your Fresh delivery is planned for 10:35 to 11:20. It may arrive after your 08:00 cut-off, so the driver will call you on the way. Please keep the receiving bay free."
OUT012,red,"Your Fresh delivery is planned for 11:25 to 12:15. It may arrive after your 08:00 cut-off, so the driver will call you on the way. Please keep the receiving bay free."


The planned arrival times on this route are the ones Waypoint uses today. The likely arrival ranges are what the route simulation expects once depot delay, real drive times and real unloading times are counted. The gap between the two columns is the reason the route is red.